In [ ]:
from pykafka import KafkaClient
import requests

client = KafkaClient(hosts="localhost:9092")
topic = client.topics[b'bitcoin_prices']

with topic.get_sync_producer() as producer:
    while True:
        price = requests.get('https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd').json()['bitcoin']['usd']
        producer.produce(f'{time.time()},{price}'.encode())
        time.sleep(1)


In [ ]:
consumer = topic.get_balanced_consumer(
    consumer_group=b'price_analyzers',
    auto_commit_enable=True,
    zookeeper_connect='localhost:2181'
)

while True:
    msg = consumer.consume()
    timestamp, price = msg.value.decode().split(',')
    process_message(float(price))


In [ ]:
from collections import deque

window_size = 60  # 1-minute window at 1Hz data
price_buffer = deque(maxlen=window_size)

def process_message(price):
    price_buffer.append(price)
    if len(price_buffer) == window_size:
        moving_avg = sum(price_buffer)/window_size
        current_deviation = price - moving_avg
        trigger_alert_if(abs(current_deviation) > 2*std_dev)


In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1)
fig.append_trace(go.Scatter(x=timestamps, y=prices), row=1, col=1)
fig.append_trace(go.Scatter(x=timestamps, y=moving_averages), row=2, col=1)
fig.show()
